# transformer-vs-ssm — unattended Kaggle trainer

**Run mode:** Save Version → *Save & Run All (Commit)*. Do not run cell-by-cell.

This notebook trains **one architecture per version** and picks which one by itself:
it reads a manifest from your Hugging Face repo, finds the first arch that hasn't
reached `TARGET_STEPS`, and trains that one (resuming if a checkpoint exists).
Hit *Save Version* again to advance the queue.

**Durability.** A background thread uploads `checkpoints/<arch>/latest.pt` to Hugging
Face every few minutes. If this session crashes, OOMs, or hits the 12h wall, the
weights are already off-box and the next run resumes from them. Nothing in this
notebook asserts before the artifact is saved — a training failure is recorded and
reported, not raised.

---

## One-time setup

1. **HF token** — huggingface.co → Settings → Access Tokens → New token, type
   *Write*. Copy it.
2. **HF repo** — create a model repo, e.g. `your-name/transformer-vs-ssm-ckpts`.
   Private is fine. (The notebook will also create it for you if missing.)
3. **Kaggle secret** — in the notebook editor: *Add-ons → Secrets → Add a secret*,
   label exactly `HF_TOKEN`, value = the token. Attach it to this notebook.
4. **Session settings** — Accelerator `GPU T4 x1`, Internet **ON**, Persistence ON.

Then set `HF_REPO` in the config cell below and commit.

In [ ]:
# Cell 1 — clone repo, set working directory
import os, sys, subprocess, time

NB_START = time.time()

if not os.path.exists('transformer-vs-ssm'):
    subprocess.run(['git', 'clone',
                    'https://github.com/nvaidyan1/transformer-vs-ssm.git'], check=True)

os.chdir('transformer-vs-ssm')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

subprocess.run(['git', 'pull'], check=False)
print('cwd:', os.getcwd())

In [ ]:
# Cell 2 — dependencies
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'torch', 'numpy', 'pyyaml', 'matplotlib', 'seaborn', 'tqdm',
     'huggingface_hub'],
    check=True
)
print('deps OK')

In [ ]:
# Cell 3 — CONFIG. Edit HF_REPO, then leave the rest alone unless you mean it.
HF_REPO = 'your-hf-username/transformer-vs-ssm-ckpts'   # <-- CHANGE THIS

# 2 epochs of enwik8: 90,000,000 bytes / (batch 16 * seq_len 1024) = 5,493 steps/epoch
TARGET_STEPS = 11_000
SAVE_EVERY   = 500        # local checkpoint cadence (steps)
LOG_EVERY    = 500        # eval + log cadence. Each eval is a FULL pass over the
                          # 10M-byte val set (~611 batches, ~115s) -- roughly 29%
                          # of wall clock at 500. Set 1000 to halve that cost.
PUSH_EVERY_S = 420        # background upload cadence (seconds)

# Kaggle kills GPU sessions at 12h. Stop cleanly before that.
SESSION_BUDGET_S = int(10.5 * 3600)

# Job queue, in order. Set enabled=False to hold an arch out of the queue.
JOBS = [
    {'arch': 'transformer', 'config': 'configs/transformer.yaml', 'enabled': True},
    {'arch': 'tcn',         'config': 'configs/tcn.yaml',         'enabled': True},
    {'arch': 'mamba',       'config': 'configs/mamba.yaml',       'enabled': True},
]

# --- HF token from Kaggle Secrets -------------------------------------------
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print(f'!! No HF_TOKEN secret ({type(e).__name__}). '
          f'Running LOCAL-ONLY: no off-box saves, no resume across sessions.')

REMOTE = HF_TOKEN is not None and not HF_REPO.startswith('your-hf-username')
if HF_TOKEN and not REMOTE:
    print('!! HF_REPO still has the placeholder value — remote saving disabled.')

import shutil
print(f'free disk: {shutil.disk_usage("/kaggle/working").free / 1024**3:.1f} GB')

In [ ]:
# Cell 4 — prepare data (no-op if already present)
from src.data import prepare_data
prepare_data()

In [ ]:
# Cell 5 — Hugging Face helpers
import json, tempfile
from pathlib import Path

MANIFEST = 'manifest.json'
_api = None

if REMOTE:
    from huggingface_hub import HfApi, hf_hub_download, create_repo
    _api = HfApi(token=HF_TOKEN)
    create_repo(HF_REPO, repo_type='model', private=True,
                exist_ok=True, token=HF_TOKEN)
    print(f'remote: https://huggingface.co/{HF_REPO}')


def hf_put(local_path, remote_path):
    # Upload one file, overwriting. Returns True on success.
    if not REMOTE:
        return False
    try:
        _api.upload_file(path_or_fileobj=str(local_path), path_in_repo=remote_path,
                         repo_id=HF_REPO, repo_type='model')
        return True
    except Exception as e:
        print(f'  [push failed] {remote_path}: {type(e).__name__}: {e}')
        return False


def hf_get(remote_path, local_path):
    # Download one file. Returns True if it landed, False if absent/failed.
    if not REMOTE:
        return False
    try:
        got = hf_hub_download(repo_id=HF_REPO, filename=remote_path,
                              repo_type='model', token=HF_TOKEN)
        Path(local_path).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(got, local_path)
        return True
    except Exception as e:
        print(f'  [pull miss] {remote_path}: {type(e).__name__}')
        return False


def load_manifest():
    if REMOTE:
        tmp = Path(tempfile.gettempdir()) / MANIFEST
        if hf_get(MANIFEST, tmp):
            try:
                return json.loads(tmp.read_text())
            except Exception:
                print('  [manifest unreadable — starting fresh]')
    return {}


def save_manifest(m):
    tmp = Path(tempfile.gettempdir()) / MANIFEST
    tmp.write_text(json.dumps(m, indent=2, sort_keys=True))
    hf_put(tmp, MANIFEST)


print('helpers ready')

In [ ]:
# Cell 6 — pick the job. First arch below TARGET_STEPS wins.
manifest = load_manifest()
print('manifest:', json.dumps(manifest, indent=2) if manifest else '(empty)')

# Guard: the uncapped dilation = 2**i schedule in tcn.py allocates a 12.02 GiB
# pad at layer 18 and OOMs instantly. Refuse to burn a session on it.
TCN_FIXED = 'dilation_cycle' in open('src/models/tcn.py').read()
if not TCN_FIXED:
    print('!! src/models/tcn.py has no dilation_cycle -- TCN held out of queue.')
    print('   Push the fixed tcn.py to the repo first, then re-commit.')

job = None
for j in JOBS:
    if not j['enabled']:
        print(f"skip {j['arch']}: disabled in JOBS")
        continue
    if j['arch'] == 'tcn' and not TCN_FIXED:
        continue
    if not os.path.exists(j['config']):
        print(f"skip {j['arch']}: {j['config']} not found in repo")
        continue
    done = manifest.get(j['arch'], {}).get('step', 0)
    if done >= TARGET_STEPS:
        print(f"skip {j['arch']}: already at step {done:,} >= {TARGET_STEPS:,}")
        continue
    job = j
    print(f"\n>>> SELECTED: {j['arch']} (resuming from step {done:,})")
    break

if job is None:
    print('\nNothing left to train. Every queued arch has reached the target.')

In [ ]:
# Cell 8 — pull any existing checkpoint so train.py auto-resumes
CKPT_DIR = Path(f"checkpoints/{job['arch']}") if job else None

if job:
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    local = CKPT_DIR / 'latest.pt'

    # train.py writes latest.pt as a SYMLINK to ckpt_NNNNNNN.pt. A dangling
    # symlink reports exists()==False, and copyfile() through it would write to
    # the missing target name instead of latest.pt. Clear it first.
    if local.is_symlink() and not local.exists():
        print('clearing dangling latest.pt symlink')
        local.unlink()

    if local.exists():
        print(f'local checkpoint already present: {local}')
    elif hf_get(f"{job['arch']}/latest.pt", local):
        print(f'pulled remote checkpoint -> {local}')
    else:
        print('no checkpoint found — training from scratch')

    if local.exists():
        import torch
        try:
            _c = torch.load(local, map_location='cpu', weights_only=False)
            print(f"  resume point: step {_c.get('step', '?'):,} "
                  f"val_bpc {_c.get('val_bpc', float('nan')):.4f}")
            del _c
        except Exception as e:
            print(f'  !! checkpoint unreadable ({e}) — moving it aside')
            local.rename(local.with_suffix('.pt.corrupt'))

In [ ]:
# Cell 9 — background uploader: watches latest.pt, pushes on change, prunes old files
import threading

_stop = threading.Event()
_pushes = {'n': 0, 'last_step': None}


def _stable_copy(src, dst, settle=3.0):
    # Copy only if the file is not mid-write (mtime stable across a short wait).
    m1 = os.path.getmtime(src)
    time.sleep(settle)
    if os.path.getmtime(src) != m1:
        return False
    shutil.copyfile(src, dst)
    return True


# NOTE: no pruning here. train.py's save_checkpoint() already evicts all but
# the 2 most recent ckpt_NNNNNNN.pt files. Deleting from this thread as well
# would race against the symlink it just repointed.


def _watch(arch, ckpt_dir, interval):
    latest = Path(ckpt_dir) / 'latest.pt'
    staged = Path(tempfile.gettempdir()) / f'{arch}_stage.pt'
    seen = 0.0
    while not _stop.wait(interval):
        try:
            if not latest.exists():
                continue
            mtime = os.path.getmtime(latest)
            if mtime <= seen:
                continue
            if not _stable_copy(latest, staged):
                continue
            if hf_put(staged, f'{arch}/latest.pt'):
                seen = mtime
                _pushes['n'] += 1
                mb = staged.stat().st_size / 1024**2
                print(f'  [push #{_pushes["n"]}] {arch}/latest.pt ({mb:.0f} MB)',
                      flush=True)
        except Exception as e:
            print(f'  [watcher] {type(e).__name__}: {e}', flush=True)


watcher = None
if job and REMOTE:
    watcher = threading.Thread(target=_watch, daemon=True, args=(
        job['arch'], CKPT_DIR, PUSH_EVERY_S))
    watcher.start()
    print(f'uploader running every {PUSH_EVERY_S}s')
elif job:
    print('uploader disabled (no remote) — checkpoints stay local only')

In [ ]:
# Cell 10 — train. Never raises: outcome is recorded, not asserted.
status = {'arch': job['arch'] if job else None, 'outcome': 'skipped',
          'returncode': None, 'seconds': 0}

if job:
    remaining = SESSION_BUDGET_S - (time.time() - NB_START)
    print(f'=== {job["arch"].upper()} TRAINING START '
          f'(budget {remaining/3600:.2f}h) ===', flush=True)

    env = dict(os.environ, PYTORCH_ALLOC_CONF='expandable_segments:True')
    t0 = time.time()
    try:
        cmd = [sys.executable, '-u', 'src/train.py',
               '--config', job['config'],
               '--max_steps', str(TARGET_STEPS),
               '--save_every', str(SAVE_EVERY),
               '--log_every', str(LOG_EVERY)]
        print('  ' + ' '.join(cmd), flush=True)
        r = subprocess.run(cmd, check=False,
                           timeout=max(remaining, 60), env=env)
        status['returncode'] = r.returncode
        status['outcome'] = 'completed' if r.returncode == 0 else 'failed'
    except subprocess.TimeoutExpired:
        status['outcome'] = 'timeout'
        print('\n!! wall-clock budget reached — training stopped, '
              'checkpoint will be pushed and the next run resumes', flush=True)
    except Exception as e:
        status['outcome'] = 'error'
        status['returncode'] = f'{type(e).__name__}: {e}'

    status['seconds'] = time.time() - t0
    print(f'=== {job["arch"].upper()} {status["outcome"].upper()} '
          f'in {status["seconds"]/3600:.2f}h (rc={status["returncode"]}) ===')

In [ ]:
# Cell 11 — stop the watcher, push final full + slim checkpoints, update manifest
_stop.set()
if watcher:
    watcher.join(timeout=180)

final_step, final_bpc = None, None

if job and (CKPT_DIR / 'latest.pt').exists():
    import torch
    ck = torch.load(CKPT_DIR / 'latest.pt', map_location='cpu', weights_only=False)
    final_step = int(ck.get('step', 0))
    final_bpc = float(ck.get('val_bpc', float('nan')))
    print(f"final: step {final_step:,} | val_bpc {final_bpc:.4f}"
          if final_bpc == final_bpc else f"final: step {final_step:,} | val_bpc n/a")

    # full checkpoint (weights + optimizer) — this is what resume needs
    hf_put(CKPT_DIR / 'latest.pt', f"{job['arch']}/latest.pt")

    # slim checkpoint (weights only) — ~1/3 the size, for eval and sharing
    # train.py's save_checkpoint writes exactly these keys:
    #   model_state, optimizer_state, config, step, val_bpc, seed
    # Drop optimizer_state only -- that is the 2/3 of the file that resume needs
    # but evaluation does not.
    slim = {k: v for k, v in ck.items() if k != 'optimizer_state'}
    assert 'model_state' in slim, 'unexpected checkpoint layout'
    slim_path = CKPT_DIR / 'slim.pt'
    torch.save(slim, slim_path)
    print(f'slim: {slim_path.stat().st_size / 1024**2:.0f} MB '
          f'(full: {(CKPT_DIR / "latest.pt").stat().st_size / 1024**2:.0f} MB)')
    hf_put(slim_path, f"{job['arch']}/slim.pt")

    def _finite(v):
        # train.py initialises last_val_bpc to inf; if the run dies before the
        # first eval that lands in the checkpoint, and json.dumps would emit a
        # bare `Infinity`, which is not strict JSON.
        try:
            v = float(v)
            return v if v == v and abs(v) != float('inf') else None
        except (TypeError, ValueError):
            return None

    manifest[job['arch']] = {
        'step': final_step,
        'val_bpc': _finite(final_bpc),
        'target': TARGET_STEPS,
        'done': final_step >= TARGET_STEPS,
        'last_outcome': status['outcome'],
        'hours': round(status['seconds'] / 3600, 2),
    }
    save_manifest(manifest)
    print('manifest updated')
elif job:
    print('!! no checkpoint on disk — training died before the first save')

In [ ]:
# Cell 12 — stage output for the Kaggle Output tab (always runs, never asserts)
import zipfile

out = Path('/kaggle/working/checkpoints.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arch_dir in sorted(Path('checkpoints').glob('*')):
        for name in ('latest.pt', 'slim.pt'):
            f = arch_dir / name
            if f.exists():
                zf.write(f, f'{arch_dir.name}/{name}')

print(f'{out} ({out.stat().st_size / 1024**2:.0f} MB)')

print('\n' + '=' * 56)
print('  RUN SUMMARY')
print('=' * 56)
print(f"  arch     : {status['arch']}")
print(f"  outcome  : {status['outcome']}")
print(f"  step     : {final_step:,}" if final_step else '  step     : n/a')
print(f"  val_bpc  : {final_bpc:.4f}" if final_bpc is not None else '  val_bpc  : n/a')
print(f"  pushes   : {_pushes['n']}")
print(f"  elapsed  : {(time.time() - NB_START)/3600:.2f}h")
print('=' * 56)

if status['outcome'] == 'completed' and final_step and final_step >= TARGET_STEPS:
    print('\nNEXT: Save Version again — the notebook will pick the next arch.')
elif status['outcome'] == 'timeout':
    print('\nNEXT: Save Version again — same arch resumes from the pushed checkpoint.')
elif status['outcome'] == 'failed':
    print('\nNEXT: read the traceback above. Nothing was lost; fix and re-commit.')